In [1]:
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
import shutil
import numpy as np
import random

random.seed(42)

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2")
DST = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2_aug")

SRC_TRAIN = SRC / "train"
SRC_VAL = SRC / "val"

DST_TRAIN = DST / "train"
DST_VAL = DST / "val"

DST_TRAIN.mkdir(parents=True, exist_ok=True)
DST_VAL.mkdir(parents=True, exist_ok=True)

print("Source exists:", SRC.exists())
print("Train exists:", SRC_TRAIN.exists())
print("Val exists:", SRC_VAL.exists())
print("Destination ready:", DST.exists())

Source exists: True
Train exists: True
Val exists: True
Destination ready: True


In [2]:
train_classes = [p.name for p in SRC_TRAIN.iterdir() if p.is_dir()]
val_classes = [p.name for p in SRC_VAL.iterdir() if p.is_dir()]

print("Train classes:", train_classes)
print("Val classes:", val_classes)
print("Number of train classes:", len(train_classes))
print("Number of val classes:", len(val_classes))

Train classes: ['01-minor', '02-moderate', '03-severe']
Val classes: ['01-minor', '02-moderate', '03-severe']
Number of train classes: 3
Number of val classes: 3


In [3]:
def aug_brightness(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img: Image.Image, radius: float = 1.5) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius))

def aug_noise(img: Image.Image, noise_level: int = 12) -> Image.Image:
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-noise_level, noise_level + 1, arr.shape, dtype=np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_shadow(img: Image.Image, factor: float = 0.7) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

In [4]:
val_copy_count = 0

for class_dir in SRC_VAL.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_VAL / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        shutil.copy2(img_path, dst_class_dir / img_path.name)
        val_copy_count += 1

print("Validation images copied:", val_copy_count)

Validation images copied: 248


In [5]:
train_copy_count = 0

for class_dir in SRC_TRAIN.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_TRAIN / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        shutil.copy2(img_path, dst_class_dir / img_path.name)
        train_copy_count += 1

print("Original training images copied:", train_copy_count)

Original training images copied: 1383


In [6]:
aug_count = 0

for class_dir in SRC_TRAIN.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_TRAIN / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        stem = img_path.stem
        suffix = img_path.suffix

        img = Image.open(img_path).convert("RGB")

        augmentations = [
            ("bright", aug_brightness(img, 1.25)),
            ("dark", aug_brightness(img, 0.75)),
            ("contrast", aug_contrast(img, 1.3)),
            ("blur", aug_blur(img, 1.5)),
            ("noise", aug_noise(img, 12)),
            ("shadow", aug_shadow(img, 0.65)),
        ]

        for tag, aug_img in augmentations:
            new_name = f"{stem}_{tag}{suffix}"
            aug_img.save(dst_class_dir / new_name)
            aug_count += 1

print("Augmented images created:", aug_count)

Augmented images created: 8298


In [7]:
final_train_count = 0
final_val_count = 0

for class_dir in DST_TRAIN.iterdir():
    if class_dir.is_dir():
        final_train_count += len(list(class_dir.glob("*.*")))

for class_dir in DST_VAL.iterdir():
    if class_dir.is_dir():
        final_val_count += len(list(class_dir.glob("*.*")))

print("Final train image count:", final_train_count)
print("Final val image count:", final_val_count)

Final train image count: 9681
Final val image count: 248


In [1]:
from pathlib import Path

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2")

for split in ["train", "val"]:
    split_dir = SRC / split
    print(f"\n=== {split.upper()} ===")
    total = 0

    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            count = len(list(class_dir.glob("*.*")))
            total += count
            print(class_dir.name, ":", count)

    print("Total:", total)


=== TRAIN ===
01-minor : 452
02-moderate : 463
03-severe : 468
Total: 1383

=== VAL ===
01-minor : 82
02-moderate : 75
03-severe : 91
Total: 248


In [2]:
from ultralytics import YOLO
from pathlib import Path
import shutil

MODEL_PATH = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\notebooks\03_training\runs\classify\archive2_yolo11m_cls_aug\weights\best.pt")
VAL_DIR = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2_aug\val")
OUT_DIR = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\reports\archive2_error_analysis")

OUT_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(str(MODEL_PATH))

wrong_count = 0
total_count = 0

for true_class_dir in VAL_DIR.iterdir():
    if not true_class_dir.is_dir():
        continue

    true_class = true_class_dir.name

    for img_path in true_class_dir.glob("*.*"):
        total_count += 1

        result = model.predict(str(img_path), verbose=False)[0]
        pred_index = int(result.probs.top1)
        pred_class = model.names[pred_index]

        if pred_class != true_class:
            wrong_count += 1

            save_dir = OUT_DIR / f"TRUE_{true_class}__PRED_{pred_class}"
            save_dir.mkdir(parents=True, exist_ok=True)

            shutil.copy2(img_path, save_dir / img_path.name)

print("Total validation images checked:", total_count)
print("Wrong predictions:", wrong_count)
print("Saved wrong images to:", OUT_DIR)

Total validation images checked: 248
Wrong predictions: 70
Saved wrong images to: C:\Users\User\Desktop\Vehicle_Damage_Detection\reports\archive2_error_analysis


In [ ]:
from pathlib import Path

ERROR_DIR = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\reports\archive2_error_analysis")

decision_folders = [
    "KEEP_TRUE_LABEL",
    "RELABEL_TO_PREDICTED",
    "REMOVE_AMBIGUOUS"
]

for folder in ERROR_DIR.iterdir():
    if folder.is_dir() and folder.name.startswith("TRUE_"):
        for decision in decision_folders:
            (folder / decision).mkdir(exist_ok=True)

print("Review folders created successfully.")